In [1]:
pip install huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from huggingface_hub import login
login(token='huggingface_token_load_from_secrets')

In [3]:
pip install datasets evaluate evaluate accelerate bitsandbytes trl -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 MB 173.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 43.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 97.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 198.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 180.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 51.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 142.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 191.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 140.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 120.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import pandas as pd
import re
import numpy as np
import string
from nltk.corpus import stopwords
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfTransformer,TfidfVectorizer
from sklearn.pipeline import Pipeline
import evaluate

In [5]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from trl import SFTTrainer

In [6]:
import transformers

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [8]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [9]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))

trainable model parameters: 3212749824
all model parameters: 3212749824
percentage of trainable model parameters: 100.00%


In [10]:
import pandas as pd

# Load the Parquet file
df = pd.read_parquet("hf://datasets/arbml/CIDAR/data/train-00000-of-00001-b2881e1b9f14c3b1.parquet")

# Display the first few rows
df.head()

,output,instruction,index
0,| الضمير | الماضي | المضارع | اسم الفاعل | اسم...,صغ من الفعل الاجوف أجَارَ اسم الفاعل و اسم ال...,9830
1,قالَ السَماءُ كَئيبَةٌ وَتَجَهَّما\r\nقُلتُ اِ...,أنشئ قصيدة مكونة من 5 أبيات، تتحدث عن السماء.,5659
2,التبرع بالكتب القديمة إلى المكتبة هو وسيلة مم...,اشرح لي لماذا يجب على شخص يريد إعادة تدوير كت...,8762
3,كان حدثًا مؤثرًا حيث تجمع مئات الأشخاص للمشار...,أنشئ سردًا لوصف حدث إطلاق البالونات.,114
4,"الأغنية هي ""اشوفك كل يوم"" من كلمات: ابراهيم خ...",ساعد المستخدم في التعرف على عنوان الأغنية من ...,6236


In [11]:
# Calculate the maximum length of text in the 'instruction' column
max_length_instruction = df['instruction'].apply(len).mean()

# Calculate the maximum length of text in the 'output' column
max_length_output = df['output'].apply(len).mean()

# Print the results
print(f"Maximum length of 'instruction': {max_length_instruction}")
print(f"Maximum length of 'output': {max_length_output}")

Maximum length of 'instruction': 60.5636
Maximum length of 'output': 306.0469


In [12]:
def tokenize_function(row):
    # Tokenize the conversations
    question = ' '.join(row["instruction"]) if isinstance(row["instruction"], list) else row["instruction"]

    row['input_ids'] = tokenizer(question, padding="max_length", truncation=True, max_length = 128, return_tensors="pt").input_ids[0]
    
    # Assuming "answer" column is already a string, no need for conversion
    row['labels'] = tokenizer(row["output"], padding="max_length", truncation=True, max_length = 256, return_tensors="pt").input_ids[0]
    
    return row


# Tokenize the DataFrame
tokenized_df = df.apply(tokenize_function, axis=1)

In [13]:
# Convert columns to list
tokenized_df['input_ids'] = tokenized_df['input_ids'].apply(lambda x: x.tolist())
tokenized_df['labels'] = tokenized_df['labels'].apply(lambda x: x.tolist())

In [14]:
tokenized_df

,output,instruction,index,input_ids,labels
0,| الضمير | الماضي | المضارع | اسم الفاعل | اسم...,صغ من الفعل الاجوف أجَارَ اسم الفاعل و اسم ال...,9830,"[128000, 42693, 82878, 64337, 101292, 102036, ...","[128000, 91, 114122, 10386, 94643, 765, 54579,..."
1,قالَ السَماءُ كَئيبَةٌ وَتَجَهَّما\r\nقُلتُ اِ...,أنشئ قصيدة مكونة من 5 أبيات، تتحدث عن السماء.,5659,"[128000, 100822, 33890, 100648, 115315, 107974...","[128000, 105448, 29010, 100649, 106202, 99819,..."
2,التبرع بالكتب القديمة إلى المكتبة هو وسيلة مم...,اشرح لي لماذا يجب على شخص يريد إعادة تدوير كت...,8762,"[128000, 13258, 101512, 30925, 107296, 57894, ...","[128000, 96057, 84927, 24102, 100700, 32173, 1..."
3,كان حدثًا مؤثرًا حيث تجمع مئات الأشخاص للمشار...,أنشئ سردًا لوصف حدث إطلاق البالونات.,114,"[128000, 100822, 33890, 100648, 119541, 101333...","[128000, 102037, 103832, 85632, 101333, 106400..."
4,"الأغنية هي ""اشوفك كل يوم"" من كلمات: ابراهيم خ...",ساعد المستخدم في التعرف على عنوان الأغنية من ...,6236,"[128000, 60942, 110705, 107411, 106829, 78373,...","[128000, 100461, 82878, 106364, 104380, 330, 1..."
...,...,...,...,...,...
9995,قالت وزارة الخارجية العراقية، في بيان نشرته عب...,لماذا امتنع العراق وتونس عن التصويت لصالح القر...,5426,"[128000, 100615, 119463, 100635, 102017, 24102...","[128000, 28590, 104211, 111981, 26957, 124024,..."
9996,الأصليين من الأعمال التي تحتاج ان تشاهد اكثر م...,اكتب مراجعة عن فيلم الاصليين.,5941,"[128000, 101052, 100936, 122433, 101454, 10092...","[128000, 113672, 42693, 100576, 100327, 64337,..."
9997,الدول التي كانت أعضاء في قوى المحور في الحرب ...,ما هي الدول التي كانت أعضاء في قوى المحور في ...,3707,"[128000, 101237, 104380, 108990, 102626, 10676...","[128000, 108990, 102626, 106766, 106173, 10740..."
9998,عندما نريد معرفة الفرق بين الاستعارة التصريحية...,مالفرق بين الإستعارة التصريحية والمكنية؟,10113,"[128000, 110368, 100887, 28590, 103260, 101169...","[128000, 24102, 100274, 100653, 51343, 108639,..."


In [15]:
from datasets import Dataset

# Assuming `tokenized_df` is your pandas DataFrame
dataset = Dataset.from_pandas(tokenized_df[:10000])

In [16]:
dataset

Dataset({
    features: ['output', 'instruction', 'index', 'input_ids', 'labels'],
    num_rows: 10000
})

In [17]:
tokenized_datasets = dataset.map(tokenize_function)# batched=True, # batch_size=...
tokenized_datasets = tokenized_datasets.remove_columns(['instruction','output'])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [18]:
pip install tensorboard

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 195.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [19]:
from trl import SFTConfig, SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling
# Set supervised fine-tuning parameters
#training_arguments = SFTConfig(dataset_text_field="text")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="pt"),
    args = SFTConfig(
            output_dir="./results",
            num_train_epochs=1,
            per_device_train_batch_size=4,
            # per_device_eval_batch_size=1,
            gradient_accumulation_steps=1,
        #     evaluation_strategy="epoch",
            optim="paged_adamw_32bit",
            save_steps=100,
            logging_steps=100,
            learning_rate=2e-5,
            weight_decay=0.001,
            fp16=False,
            bf16=True,
            max_grad_norm=0.3,
        #     max_steps=-1,
            warmup_ratio=0.03,
            group_by_length=True,
            lr_scheduler_type="cosine",
            report_to="tensorboard",
            packing=False,
            dataset_text_field="text",
            max_seq_length=None,

    ),
)

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
100,3.281500
200,2.742900
300,2.617300
400,2.549500
500,2.511400
600,2.484400
700,2.489800
800,2.372100
900,2.344200
1000,2.314000


TrainOutput(global_step=2500, training_loss=2.358807995605469, metrics={'train_runtime': 9752.1209, 'train_samples_per_second': 1.025, 'train_steps_per_second': 0.256, 'total_flos': 2.164797997056e+16, 'train_loss': 2.358807995605469})

In [21]:
trainer.model.save_pretrained("./llama-32-3B-finetune-Arabic")
tokenizer.save_pretrained("./llama-32-3B-finetune-Arabic")

('./llama-32-3B-finetune-Arabic/tokenizer_config.json',
 './llama-32-3B-finetune-Arabic/special_tokens_map.json',
 './llama-32-3B-finetune-Arabic/tokenizer.json')

In [26]:
def single_inference(question):
    messages = [
        {"role": "system", "content": "اجب علي الاتي بالعربي فقط."},
    ]

    messages.append({"role": "user", "content": question})


    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.1,
    #     top_p=0.9,
    )
    response = outputs[0][input_ids.shape[-1]:]
    output = tokenizer.decode(response, skip_special_tokens=True)
    return output

In [27]:
question = """ما هي طريقة عمل البيتزا , اجب في خطوات"""

answer = single_inference(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


INPUT QUESTION:
ما هي طريقة عمل البيتزا , اجب في خطوات


Model Answer:
طريقة عمل البيتزا في خطوات هي:

1.  إعداد المكونات: البيتزا تتكون من البيض، السلطة، والجبن، والفلفل، والثوم، والفطر، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول السوداني، والفول
